# Polling Data Analysis of Climate Ads

This notebook examines whether **localized (contextualized)** climate change messaging leads to higher public concern (measured by “Yes” responses) compared to **generic** messaging.

## Research Questions

1. **Does localized (contextualized) climate change messaging increase public concern about climate change?**  
2. **How do demographic factors (location, age, and gender) impact this effect?**

We will use **A/B testing** (two‐proportion z‐tests) and **Bayesian approach** to see if there is a statistically significant difference in “Yes” responses between the two ad types (Contextualized vs. Generic).


## 1. Setup: Imports & Data Loading

In [ ]:
# # 1. Setup: Imports & Data Loading
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest

# For inline plotting in Jupyter:
%matplotlib inline

In [ ]:
# Load the Excel file
data = pd.read_excel("data/Climate_Perception_results_Part1.xlsx")

# Standardize column names
data.columns = data.columns.str.strip()
print("Column Names:", data.columns.tolist())

In [ ]:
# Function to parse Adset_Name
def parse_adset_name(adset_name):
    pattern = r"Climate_Perceptions_(.*?)_(male|female)_\((\d+, \d+)\)_(Contextualized|Generic)_Adset"
    match = re.search(pattern, adset_name)
    if match:
        location, gender, age_group, target = match.groups()
        return location, gender.capitalize(), age_group.replace(", ", "-"), target
    return None, None, None, None

# Apply parsing function
data[['Location', 'Gender', 'Age_Group', 'Target']] = data['Adset_Name'].apply(lambda x: pd.Series(parse_adset_name(x)))

# Ensure Response column is string and strip spaces
data['Response'] = data['Response'].astype(str).str.strip()

# Extract numerical values from the Response column with improved regex
data['Yes_Responses'] = data['Response'].str.extract(r'[Yy]:\s*(\d+)').astype(int)
data['No_Responses'] = data['Response'].str.extract(r'[Nn]:\s*(\d+)').astype(int)

# Convert Reach column to numerical (assuming it's in thousands format)
data['Reach_k'] = data['Reach'] * 1000

# Select relevant columns
df = data[['Location', 'Gender', 'Age_Group', 'Target', 'Reach_k', 'Yes_Responses', 'No_Responses']]

In [ ]:
# Display the DataFrame
print("=== RAW DATAFRAME ===")
display(df)


## 2. Exploratory Data Analysis

Let's compute some quick statistics to understand the dataset better, including:
- Total poll responses
- Participation rates
- Yes/No proportions


In [ ]:
# Create derived columns for easier analysis
df['Total_Responses'] = df['Yes_Responses'] + df['No_Responses']
df['Participation_Rate_%'] = (df['Total_Responses'] / df['Reach_k']) * 100
df['Yes_Rate_%'] = (df['Yes_Responses'] / df['Total_Responses']) * 100

print("=== DATA WITH DERIVED METRICS ===")
display(df)

In [ ]:
# Basic descriptive stats grouped by 'Target'
desc = df.groupby('Target')[['Yes_Responses','No_Responses','Total_Responses']].sum()
desc['Yes_Percent'] = desc['Yes_Responses'] / desc['Total_Responses'] * 100
desc['No_Percent']  = desc['No_Responses'] / desc['Total_Responses']  * 100

print("\n=== AGGREGATE STATS BY TARGET ===")
display(desc)

In [ ]:
# Bar plot of Yes_Rate_% by row
plt.figure(figsize=(10,5))
sns.barplot(x=df.index, y='Yes_Rate_%', hue='Target', data=df)
plt.title("Yes_Rate_% by Row (Contextualized vs. Generic)")
plt.xlabel("Index in DataFrame")
plt.ylabel("Yes Rate (%)")
plt.show()

In [ ]:
# Compare Contextualized vs. Generic for subgroups where both exist
subgroups_to_compare = df.query(
    "(Location=='Cologne' & Gender=='Male' & Age_Group=='50-65') | \
     (Location=='Blieskastel' & Gender=='Female' & Age_Group=='25-49')"
).copy()

# Create a more convenient ID for each subgroup
subgroups_to_compare['Subgroup'] = subgroups_to_compare['Location'] + '_' + subgroups_to_compare['Gender'] + '_' + subgroups_to_compare['Age_Group']

plt.figure(figsize=(8,5))
sns.barplot(
    x='Subgroup',
    y='Yes_Rate_%',
    hue='Target',
    data=subgroups_to_compare
)
plt.title('Subgroup-Specific Comparison: Contextualized vs. Generic')
plt.ylabel('Yes Rate (%)')
plt.xlabel('Subgroup')
plt.ylim(0, 110)
plt.legend(title='Target')
plt.tight_layout()
plt.show()

In [ ]:
# Aggregating responses by Gender
gender_summary = df.groupby("Gender")[["Yes_Responses", "No_Responses"]].sum()

# Calculate Yes response rate by gender
gender_summary["Yes_Rate"] = gender_summary["Yes_Responses"] / (gender_summary["Yes_Responses"] + gender_summary["No_Responses"])

# Bar plot for Yes response rate by gender using Matplotlib
plt.figure(figsize=(8, 6))
colors = ["blue", "orange"]
plt.bar(gender_summary.index, gender_summary["Yes_Rate"], color=colors, alpha=0.7)

# Add data labels
for i, v in enumerate(gender_summary["Yes_Rate"]):
    plt.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=12, fontweight="bold")

# Labels and title
plt.title("Proportion of 'Yes' Responses by Gender", fontsize=14)
plt.xlabel("Gender", fontsize=12)
plt.ylabel("Proportion of Yes Responses", fontsize=12)
plt.ylim(0, 1)  # Keep the scale between 0 and 1
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show the plot
plt.show()

## 3. Hypothesis Statements

We frame our A/B tests as follows:

- **Null Hypothesis (H₀)**: The proportion of “Yes” responses is the **same** for Contextualized (A) and Generic (B) ads. Mathematically,  

  p_A = p_B

- **Alternative Hypothesis (H₁)**: The proportion of “Yes” responses is **different** between the two ad types, i.e.,  

  p_A != p_B


We will run **two‐proportion z‐tests** for:
1. The subgroups where both `Contextualized` and `Generic` exist.  
2. The aggregated total across all data.


## 4. Subgroup A/B Testing

In [ ]:
# Subgroup A: Cologne (Male, 50-65)
# We compare Contextualized vs. Generic for that exact slice.

# Hard-coded from known data:
yes_cologne_context = 9
no_cologne_context  = 12

yes_cologne_generic = 4
no_cologne_generic  = 4

count_cologne = np.array([yes_cologne_context, yes_cologne_generic])
nobs_cologne  = np.array([
    yes_cologne_context + no_cologne_context,
    yes_cologne_generic + no_cologne_generic
])

stat_cologne, pval_cologne = proportions_ztest(count_cologne, nobs_cologne)

print("=== TWO-PROPORTION Z-TEST: Cologne (Male, 50-65) ===")
print(f"Contextualized: yes={yes_cologne_context}, no={no_cologne_context}")
print(f"Generic       : yes={yes_cologne_generic}, no={no_cologne_generic}")
print(f"Z-statistic   = {stat_cologne:.4f}")
print(f"p-value       = {pval_cologne:.4f}")


### Discussion: Cologne (Male, 50–65)
- The p‐value here is typically > 0.7, indicating that we **fail to reject** H₀.
- Observed “Yes” percentages: ~42.9% (Contextualized) vs. 50% (Generic).  
- With such small samples, the difference is likely due to chance.




In [ ]:
# Subgroup B: Blieskastel (Female, 25-49)
# Compare Contextualized vs. Generic for that slice.

yes_blies_f_context = 5
no_blies_f_context  = 0

yes_blies_f_generic = 4
no_blies_f_generic  = 1

count_blies_f = np.array([yes_blies_f_context, yes_blies_f_generic])
nobs_blies_f  = np.array([
    yes_blies_f_context + no_blies_f_context,
    yes_blies_f_generic + no_blies_f_generic
])

stat_blies_f, pval_blies_f = proportions_ztest(count_blies_f, nobs_blies_f)

print("=== TWO-PROPORTION Z-TEST: Blieskastel (Female, 25-49) ===")
print(f"Contextualized: yes={yes_blies_f_context}, no={no_blies_f_context}")
print(f"Generic       : yes={yes_blies_f_generic}, no={no_blies_f_generic}")
print(f"Z-statistic   = {stat_blies_f:.4f}")
print(f"p-value       = {pval_blies_f:.4f}")


### Discussion: Blieskastel (Female, 25–49)
- Observed “Yes” = 100% (5/5) for Contextualized vs. 80% (4/5) for Generic.
- The p‐value ~0.29 is **not** below 0.05, so we again **fail to reject** H₀.
- Although the raw difference is notable, the sample size (total 10) is far too small to confirm a statistically significant effect.




## 5. Aggregated A/B Test

In [ ]:

# Summing all "Yes" and "No" for each type across the entire dataset.
yes_context_all = 41
no_context_all  = 31  # => total 72

yes_generic_all = 8
no_generic_all  = 5   # => total 13

count_all = np.array([yes_context_all, yes_generic_all])
nobs_all  = np.array([
    yes_context_all + no_context_all,
    yes_generic_all + no_generic_all
])

stat_agg, pval_agg = proportions_ztest(count_all, nobs_all)

print("=== AGGREGATED TEST: ALL Contextualized vs. ALL Generic ===")
print(f"Contextualized: yes={yes_context_all}, no={no_context_all}, total={nobs_all[0]}")
print(f"Generic       : yes={yes_generic_all}, no={no_generic_all}, total={nobs_all[1]}")
print(f"Z-statistic   = {stat_agg:.4f}")
print(f"p-value       = {pval_agg:.4f}")


### Discussion: Aggregated A/B Test

- The p‐value is typically ~0.76, indicating **no statistically significant** difference across the entire data set.
- Observed “Yes” rates: Contextualized ≈ 56.9% vs. Generic ≈ 61.5%.
- Once again, the **null hypothesis** stands: we cannot rule out that the difference is due to chance.



## 6. Bayesian A/B Testing in Python

Given that the small sample size severely limits any meaningful statistical analysis, we look at alternative ways to extract value from our limited data.

We use a Beta distribution, which is commonly used in Bayesian A/B testing to model proportions. The Beta distribution (α, β) represents our belief about the probability of success (Yes responses).

A Bayesian approach allows us to estimate posterior probabilities for whether Contextualized ads are better than Generic ads. This avoids reliance on p-values and instead gives a probability distribution over possible differences.

We assume a non-informative prior, meaning α = β = 1 (flat prior).

In [ ]:
# Aggregate responses for Contextualized and Generic ads
summary = df.groupby("Target").sum()[["Yes_Responses", "No_Responses"]]

# Extract counts
yes_contextualized = summary.loc["Contextualized", "Yes_Responses"]
no_contextualized = summary.loc["Contextualized", "No_Responses"]
yes_generic = summary.loc["Generic", "Yes_Responses"]
no_generic = summary.loc["Generic", "No_Responses"]

# Define priors (Beta distributions)
alpha_prior, beta_prior = 1, 1  # Non-informative prior

# Posterior distributions for Contextualized and Generic ads
alpha_contextualized = alpha_prior + yes_contextualized
beta_contextualized = beta_prior + no_contextualized
alpha_generic = alpha_prior + yes_generic
beta_generic = beta_prior + no_generic

# Generate posterior samples
samples_contextualized = np.random.beta(alpha_contextualized, beta_contextualized, 10000)
samples_generic = np.random.beta(alpha_generic, beta_generic, 10000)

# Calculate probability that Contextualized > Generic
prob_contextualized_better = np.mean(samples_contextualized > samples_generic)
print(f"Probability Contextualized ads are better: {prob_contextualized_better:.3f}")


In [ ]:
# Plot the probability distributions for each ad type to see how they compare.

plt.figure(figsize=(10, 6))

# Plot both distributions
plt.hist(samples_contextualized, bins=50, alpha=0.6, label="Contextualized Ads", color='blue', density=True)
plt.hist(samples_generic, bins=50, alpha=0.6, label="Generic Ads", color='orange', density=True)

# Labels and legend
plt.axvline(samples_contextualized.mean(), color='blue', linestyle='dashed', label=f"Contextualized Mean: {samples_contextualized.mean():.3f}")
plt.axvline(samples_generic.mean(), color='orange', linestyle='dashed', label=f"Generic Mean: {samples_generic.mean():.3f}")
plt.xlabel("Conversion Rate (Yes Response Rate)")
plt.ylabel("Density")
plt.title("Posterior Distributions for Contextualized vs. Generic Ads")
plt.legend()
plt.show()


The value of prob_contextualized_better tells us how likely it is that Contextualized ads perform better than Generic ads.
- If p ≈ 0.5, there's no strong evidence of a difference.
- If p > 0.7, there’s moderate evidence that Contextualized is better.
- If p > 0.9, we can be fairly confident Contextualized is better.

If the Contextualized distribution is consistently to the right of the Generic distribution, it suggests it is more effective.
If there’s a lot of overlap, the difference is likely insignificant due to small sample size.

We observe that the posterior probability that Contextualized ads are better than Generic ads is 0.404 (40.4%). 

Since 0.404 is below 0.5, this suggests that Generic ads slightly outperformed Contextualized ads in terms of "Yes" responses.
However, the difference is not strong—it’s close to 50-50, meaning there's high uncertainty in the results.

The histogram distributions in the plot confirm this:
- The blue (Contextualized) and orange (Generic) distributions overlap a lot.
- The means are close: Generic (0.60) vs. Contextualized (0.569).
- This suggests that we don’t have enough evidence to confidently say one type of ad is better.

**Low sample size makes it inconclusive**

Based on our current data, we estimate a 40.4% probability that Contextualized ads outperform Generic ads. However, given the small sample size, the results remain inconclusive. The overlapping probability distributions indicate that a larger dataset is necessary to detect a meaningful difference.

# 6. Discussion

From both **subgroup** and **aggregated** A/B testing:  
- No statistically significant difference emerged between Contextualized (localized) ads and Generic ads.  
- The small sample sizes (especially in the Generic condition) limit our power to detect real differences.  

To further analyze the data beyond traditional significance testing, we applied **a Bayesian A/B testing approach**. The results showed a **40.6% probability that Contextualized ads outperform Generic ads**. This suggests that while Generic ads had a slightly higher observed success rate, the small dataset introduces **high uncertainty**, making it difficult to draw firm conclusions. The posterior distributions of the two ad types largely overlap, reinforcing the **need for more data** to detect meaningful differences.  

**Key Findings**  
1. **Cologne (Male, 50–65)** test yielded a high p‐value (~0.73), indicating no significant difference between Contextualized and Generic ads.  
2. **Blieskastel (Female, 25–49)** test also had a p‐value (~0.29), above the 0.05 threshold.  
3. **Aggregated A/B testing** across all data resulted in a p-value (~0.76), failing to reject the null hypothesis.  
4. **Bayesian analysis** revealed that the probability of Contextualized ads performing better is **only 40.6%**, suggesting that **Generic ads may have a slight edge, but with high uncertainty due to sample size constraints**.  
5. **Proportion of 'Yes' Responses by Gender**:  
   - **Female respondents showed a higher concern rate (74%)** compared to **males (53%)**.  
   - This suggests that **women may be more responsive to climate messaging**, though further research is needed to assess whether this effect holds across larger samples and different ad framings.  

**Limitations**  
- **Small total poll counts** in each subgroup, with only two subgroups exposed to Generic ads.  
- **Unbalanced conditions**: Contextualized ads received more reach, potentially skewing comparisons.  
- **Confounding factors**: Differences in ad design (e.g., visuals, background music) may have influenced engagement beyond the messaging strategy itself.  
- **Bayesian uncertainty**: While Bayesian inference provides a probability estimate rather than a binary decision, the **overlapping probability distributions indicate the need for more data to draw clearer conclusions**.  

### **Next Steps**  
- **Expand data collection**: A larger, **balanced sample** where each demographic segment receives **both ad types** with comparable reach.  
- **Control for confounding factors**: Ensure that **all ad elements except the message framing** remain **identical** to isolate the effect of localized vs. generic messaging.  
- **Further gender-based analysis**: Given the **higher engagement from female respondents**, future studies could explore **whether different messaging styles resonate differently across genders**.  

With the current dataset, we **cannot confirm** that localized ads significantly increase concern levels compared to generic ones. However, **Bayesian inference suggests high uncertainty rather than conclusive evidence that Generic ads are superior**. The **higher Yes response rate among females** hints at potential **demographic differences in engagement with climate messaging**, which future studies should investigate in a more controlled and balanced experimental setup.